# 02 Feature Engineering

This notebook builds price, volatility, volume, and optional derivatives-aware features. The `FeatureEngineer` class applies a one-period lag by default so the output is safe to use in walk-forward modeling.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.features import FeatureEngineer
from src.utils import load_yaml_config

config = load_yaml_config(PROJECT_ROOT / "config" / "parameters.yaml")
raw_dir = PROJECT_ROOT / config["data"]["raw_data_dir"]
processed_dir = PROJECT_ROOT / config["data"]["processed_data_dir"]
processed_dir.mkdir(parents=True, exist_ok=True)

spot = pd.read_csv(raw_dir / "nsei_spot.csv", index_col=0, parse_dates=True)
vix = pd.read_csv(raw_dir / "india_vix.csv", index_col=0, parse_dates=True)
engineer = FeatureEngineer(
    annualization_factor=config["features"]["annualization_factor"],
    lag_features_by=config["features"]["lag_features_by"],
)


In [ ]:
feature_frame = engineer.build_features(spot, india_vix=vix[["close"]], dropna=False)
feature_frame.to_csv(processed_dir / "nsei_features.csv")
feature_frame[[
    "close",
    "daily_return",
    "momentum_5",
    "rolling_volatility_20",
    "rsi_14",
    "adx",
    "india_vix",
]].tail(10)
